# Revision A (final) — safe to run end to end

**There is no checkpoint-deletion cell in this notebook.** The one in the
previous version destroyed the adaptive-graph checkpoints, so it is gone.

**Already measured, nothing to retrain:**

| Result | Status |
|---|---|
| Three-seed ladder rungs | done, recorded |
| Adaptive-graph backbone | done; numbers recorded, checkpoints lost |
| CTR-GCN action model | done (epoch 60) |

**Still to run:** CTR-GCN re-ID and attacks, bone stream, entropy objective,
tables. About 4.5 hours on an A100, resumable at any point.

You can now use **Run All** safely. Every training cell checks for an existing
result first and skips it.

## 0 · Setup

## 0 · Setup — mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 0 · Setup — paths and constants

In [ ]:
import os, sys, re, json, time, math, pickle, random
import numpy as np

SEED = 1
random.seed(SEED); np.random.seed(SEED)

IN_COLAB  = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = '/content/drive/MyDrive/PhD_Q1'
    SCRATCH = '/content/scratch'
elif IN_KAGGLE:
    PROJECT = '/kaggle/working/PhD_Q1'
    SCRATCH = '/kaggle/working/scratch'
else:
    PROJECT = os.path.abspath('./PhD_Q1')
    SCRATCH = os.path.abspath('./scratch')

DATA_DIR = os.path.join(PROJECT, 'data')
CKPT_DIR = os.path.join(PROJECT, 'checkpoints')
RES_DIR  = os.path.join(PROJECT, 'results')
for d in (DATA_DIR, CKPT_DIR, RES_DIR, SCRATCH):
    os.makedirs(d, exist_ok=True)

PKL_PATH  = os.path.join(DATA_DIR, 'ntu60_3danno.pkl')
CACHE_X   = os.path.join(DATA_DIR, 'ntu60_T64_x.npy')
CACHE_M   = os.path.join(DATA_DIR, 'ntu60_T64_meta.npz')

# geometry / tensor shapes
NUM_NODE, MAX_PERSON, T_LEN, IN_CH = 25, 2, 64, 3

print('project :', PROJECT)
print('pkl     :', PKL_PATH, '->', os.path.exists(PKL_PATH))

## 0 · Setup — torch and device

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', DEVICE)
if DEVICE == 'cuda':
    print(torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected. Colab: Runtime > Change runtime type > GPU.')

## 1 · Load the NTU annotations

In [ ]:
t0 = time.time()
with open(PKL_PATH, 'rb') as f:
    D = pickle.load(f)
print('loaded in %.1f s' % (time.time() - t0))

anns  = D['annotations']
names = [a['frame_dir'] for a in anns]

setup   = np.array([int(n[1:4])   for n in names], dtype=np.int16)
camera  = np.array([int(n[5:8])   for n in names], dtype=np.int16)
subject = np.array([int(n[9:12])  for n in names], dtype=np.int16)
action  = np.array([a['label']    for a in anns],  dtype=np.int16)

print('samples  :', len(anns))
print('subjects :', len(np.unique(subject)))
print('cameras  :', np.unique(camera))
print('actions  :', len(np.unique(action)))
print('example  :', names[0])
print('keys     :', list(anns[0].keys()))
print('keypoint :', np.asarray(anns[0]['keypoint']).shape, ' (M, T, V, C)')

assert len(np.unique(subject)) == 40, 'expected 40 subjects'
print('\nOK — identity labels present. Proceed.')

## 2 · Preprocessing functions

In [ ]:
def _unit(v):
    n = np.linalg.norm(v)
    return v if n < 1e-8 else v / n

def _rotation_matrix(axis, theta):
    axis = _unit(np.asarray(axis, dtype=np.float64))
    a = math.cos(theta / 2.0)
    b, c, d = -axis * math.sin(theta / 2.0)
    aa, bb, cc, dd = a*a, b*b, c*c, d*d
    bc, ad, ac, ab, bd, cd = b*c, a*d, a*c, a*b, b*d, c*d
    return np.array([[aa+bb-cc-dd, 2*(bc+ad),   2*(bd-ac)],
                     [2*(bc-ad),   aa+cc-bb-dd, 2*(cd+ab)],
                     [2*(bd+ac),   2*(cd-ab),   aa+dd-bb-cc]], dtype=np.float32)

def _angle_between(v1, v2):
    return np.arccos(np.clip(np.dot(_unit(v1), _unit(v2)), -1.0, 1.0))

def _align(skel, ja, jb, target):
    # rotate the whole sequence so that joint ja -> jb points along `target`
    vec = skel[0, 0, jb] - skel[0, 0, ja]
    if np.linalg.norm(vec) < 1e-8:
        return skel
    axis, angle = np.cross(vec, target), _angle_between(vec, target)
    if np.linalg.norm(axis) < 1e-8 or angle < 1e-8:
        return skel
    return (skel @ _rotation_matrix(axis, angle).T).astype(np.float32)

def pre_normalize(kp):
    kp = np.asarray(kp, dtype=np.float32).copy()
    valid = kp.sum(axis=(0, 2, 3)) != 0
    if valid.any():
        kp = kp[:, valid]
    if kp.shape[1] == 0:
        return np.zeros((1, 1, NUM_NODE, 3), dtype=np.float32)
    mask = kp.sum(axis=-1) != 0
    kp[mask] = kp[mask] - kp[0, 0, 1].copy()          # centre on spine
    kp = _align(kp, 0, 1, np.array([0, 0, 1], np.float32))   # spine     -> z
    kp = _align(kp, 8, 4, np.array([1, 0, 0], np.float32))   # shoulders -> x
    return kp

def resample(kp, t_len=T_LEN):
    M, T, V, Ch = kp.shape
    out = kp[:, np.linspace(0, T - 1, t_len).round().astype(np.int64)]
    if M < MAX_PERSON:
        out = np.concatenate(
            [out, np.zeros((MAX_PERSON - M, t_len, V, Ch), out.dtype)], axis=0)
    return out[:MAX_PERSON]

# quick self-test on synthetic data
_t = np.random.RandomState(0).normal(size=(1, 90, 25, 3)).astype(np.float32)
assert resample(pre_normalize(_t)).shape == (MAX_PERSON, T_LEN, NUM_NODE, 3)
print('preprocessing OK')

## 2 · Build or load the cache

In [ ]:
if os.path.exists(CACHE_X) and os.path.exists(CACHE_M):
    print('cache found, skipping preprocessing')
else:
    N = len(anns)
    X = np.zeros((N, MAX_PERSON, T_LEN, NUM_NODE, IN_CH), dtype=np.float16)
    t0 = time.time()
    for i, a in enumerate(anns):
        X[i] = resample(pre_normalize(np.asarray(a['keypoint']))).astype(np.float16)
        if (i + 1) % 5000 == 0:
            el = time.time() - t0
            print('  %6d / %d   %.0f s elapsed, ~%.0f s left'
                  % (i + 1, N, el, el / (i + 1) * (N - i - 1)))
    np.save(CACHE_X, X)
    np.savez(CACHE_M, setup=setup, camera=camera, subject=subject,
             action=action, names=np.array(names))
    print('cached in %.0f s -> %.2f GB' % (time.time() - t0, X.nbytes / 1e9))
    del X

## 2 · Copy the cache to local disk

In [ ]:
# copy to local disk first: Drive I/O is slow for repeated reads
LOCAL_X = os.path.join(SCRATCH, 'x.npy')
if not os.path.exists(LOCAL_X):
    import shutil; shutil.copy(CACHE_X, LOCAL_X)

X = np.load(LOCAL_X)                       # (N, 2, 64, 25, 3) float16
_m = np.load(CACHE_M, allow_pickle=True)
setup, camera, subject, action = _m['setup'], _m['camera'], _m['subject'], _m['action']
names = list(_m['names'])

SUBJ_IDS   = sorted(np.unique(subject).tolist())
SUBJ_LABEL = np.array([SUBJ_IDS.index(s) for s in subject], dtype=np.int64)  # 0..39
ACT_LABEL  = action.astype(np.int64)                                         # 0..59

print('X', X.shape, X.dtype, '%.2f GB' % (X.nbytes / 1e9))
print('action classes', ACT_LABEL.max() + 1, '| identity classes', SUBJ_LABEL.max() + 1)

## 3 · Evaluation protocols (splits)

In [ ]:
XSUB_TRAIN = [1,2,4,5,8,9,13,14,15,16,17,18,19,25,27,28,31,34,35,38]

SPLITS = {}
SPLITS['xsub'] = (np.where(np.isin(subject, XSUB_TRAIN))[0],
                  np.where(~np.isin(subject, XSUB_TRAIN))[0])

rs, tr, te = np.random.RandomState(SEED), [], []
for s in SUBJ_IDS:                                   # stratified 80/20 per person
    idx = np.where(subject == s)[0]; rs.shuffle(idx)
    k = int(round(0.8 * len(idx)))
    tr += idx[:k].tolist(); te += idx[k:].tolist()
SPLITS['reid_rand'] = (np.array(sorted(tr)), np.array(sorted(te)))

SPLITS['reid_xview'] = (np.where(np.isin(camera, [1, 2]))[0],
                        np.where(camera == 3)[0])

for k, (a, b) in SPLITS.items():
    print('%-11s train %6d  test %6d  | subjects %2d / %2d'
          % (k, len(a), len(b), len(set(subject[a])), len(set(subject[b]))))

assert len(set(subject[SPLITS['xsub'][0]]) & set(subject[SPLITS['xsub'][1]])) == 0
assert len(set(subject[SPLITS['reid_xview'][0]]) &
           set(subject[SPLITS['reid_xview'][1]])) == 40

## 4 · ST-GCN definition

In [ ]:
_INWARD_1B = [(1,2),(2,21),(3,21),(4,3),(5,21),(6,5),(7,6),(8,7),(9,21),
              (10,9),(11,10),(12,11),(13,1),(14,13),(15,14),(16,15),
              (17,1),(18,17),(19,18),(20,19),(22,23),(23,8),(24,25),(25,12)]
INWARD    = [(i-1, j-1) for i, j in _INWARD_1B]
OUTWARD   = [(j, i) for i, j in INWARD]
SELF_LINK = [(i, i) for i in range(NUM_NODE)]

def _edge2mat(link, n):
    A = np.zeros((n, n), dtype=np.float32)
    for i, j in link:
        A[j, i] = 1.0
    return A

def _norm_digraph(A):
    Dl, Dn = A.sum(0), np.zeros_like(A)
    for i in range(A.shape[0]):
        if Dl[i] > 0:
            Dn[i, i] = Dl[i] ** -1.0
    return A @ Dn

def build_adjacency():
    return np.stack([_edge2mat(SELF_LINK, NUM_NODE),
                     _norm_digraph(_edge2mat(INWARD, NUM_NODE)),
                     _norm_digraph(_edge2mat(OUTWARD, NUM_NODE))]).astype(np.float32)


class SpatialGC(nn.Module):
    def __init__(self, in_c, out_c, A):
        super().__init__()
        self.K = A.shape[0]
        self.conv = nn.Conv2d(in_c, out_c * self.K, 1)
        self.register_buffer('A', torch.from_numpy(A.copy()))

    def forward(self, x):
        N, _, T, V = x.shape
        x = self.conv(x).view(N, self.K, -1, T, V)
        return torch.einsum('nkctv,kvw->nctw', x, self.A.to(x.dtype)).contiguous()


class STGCNBlock(nn.Module):
    def __init__(self, in_c, out_c, A, stride=1, residual=True):
        super().__init__()
        self.gcn = SpatialGC(in_c, out_c, A)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.tcn = nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0))
        self.bn2 = nn.BatchNorm2d(out_c)
        self.drop = nn.Dropout(0.1)
        if not residual:
            self.res = None
        elif in_c == out_c and stride == 1:
            self.res = nn.Identity()
        else:
            self.res = nn.Sequential(nn.Conv2d(in_c, out_c, 1, (stride, 1)),
                                     nn.BatchNorm2d(out_c))

    def forward(self, x):
        r = 0 if self.res is None else self.res(x)
        y = F.relu(self.bn1(self.gcn(x)))
        y = self.drop(self.bn2(self.tcn(y)))
        return F.relu(y + r)


class STGCN(nn.Module):
    def __init__(self, num_class, in_channels=IN_CH, num_person=MAX_PERSON, dropout=0.3):
        super().__init__()
        A = build_adjacency()
        self.data_bn = nn.BatchNorm1d(num_person * in_channels * NUM_NODE)
        cfg = [(in_channels, 64, 1, False), (64, 64, 1, True), (64, 64, 1, True),
               (64, 64, 1, True), (64, 128, 2, True), (128, 128, 1, True),
               (128, 128, 1, True), (128, 256, 2, True), (256, 256, 1, True),
               (256, 256, 1, True)]
        self.layers = nn.ModuleList([STGCNBlock(i, o, A, s, r) for i, o, s, r in cfg])
        self.drop, self.fc, self.feat_dim = nn.Dropout(dropout), nn.Linear(256, num_class), 256

    def extract(self, x):                      # x: N, C, T, V, M
        N, C, T, V, M = x.shape
        x = x.permute(0, 4, 3, 1, 2).contiguous().view(N, M * V * C, T)
        x = self.data_bn(x)
        x = x.view(N, M, V, C, T).permute(0, 1, 3, 4, 2).contiguous().view(N * M, C, T, V)
        for l in self.layers:
            x = l(x)
        return F.adaptive_avg_pool2d(x, 1).view(N, M, -1).mean(1)

    def forward(self, x, return_feat=False):
        f = self.extract(x)
        out = self.fc(self.drop(f))
        return (out, f) if return_feat else out


_m = STGCN(60)
_o, _f = _m(torch.randn(2, IN_CH, T_LEN, NUM_NODE, MAX_PERSON), return_feat=True)
print('logits', tuple(_o.shape), '| features', tuple(_f.shape))
print('parameters: %.2f M' % (sum(p.numel() for p in _m.parameters()) / 1e6))
del _m, _o, _f

## 5 · Training engine

In [ ]:
def _euler(theta):
    cx, cy, cz = np.cos(theta); sx, sy, sz = np.sin(theta)
    Rx = np.array([[1,0,0],[0,cx,-sx],[0,sx,cx]])
    Ry = np.array([[cy,0,sy],[0,1,0],[-sy,0,cy]])
    Rz = np.array([[cz,-sz,0],[sz,cz,0],[0,0,1]])
    return (Rz @ Ry @ Rx).astype(np.float32)

def augment(x, rot=0.3, scale=0.1):
    R = _euler(np.random.uniform(-rot, rot, 3))
    s = 1.0 + np.random.uniform(-scale, scale)
    mask = (x.sum(-1) != 0)[..., None]
    return np.where(mask, (x @ R.T) * s, 0.0).astype(np.float32)


class SkeletonDS(torch.utils.data.Dataset):
    def __init__(self, X, y, idx, train=False):
        self.X, self.y, self.idx, self.train = X, y, idx, train
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        x = self.X[j].astype(np.float32)
        if self.train:
            x = augment(x)
        x = np.ascontiguousarray(np.transpose(x, (3, 1, 2, 0)))   # C,T,V,M
        return torch.from_numpy(x), int(self.y[j])


def _load_ckpt(p):
    try:
        return torch.load(p, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(p, map_location='cpu')


@torch.no_grad()
def evaluate(model, loader):
    model.eval(); correct = total = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
            pred = model(xb).argmax(1)
        correct += (pred == yb).sum().item(); total += yb.numel()
    return 100.0 * correct / total


def run_experiment(tag, labels, split, num_classes, epochs=60, lr=0.1, bs=64,
                   wd=5e-4, warmup=5, workers=2, init_from=None, freeze=False):
    tr_idx, te_idx = SPLITS[split]
    tl = torch.utils.data.DataLoader(SkeletonDS(X, labels, tr_idx, True), bs, shuffle=True,
                                     num_workers=workers, pin_memory=(DEVICE == 'cuda'),
                                     drop_last=True, persistent_workers=workers > 0)
    vl = torch.utils.data.DataLoader(SkeletonDS(X, labels, te_idx, False), bs * 2,
                                     shuffle=False, num_workers=workers,
                                     pin_memory=(DEVICE == 'cuda'))

    model = STGCN(num_classes).to(DEVICE)
    if init_from:
        sd = _load_ckpt(init_from)['model']
        sd = {k: v for k, v in sd.items() if not k.startswith('fc.')}
        print('  init from', os.path.basename(init_from),
              model.load_state_dict(sd, strict=False).missing_keys[:2], '...')
    if freeze:
        for n_, p_ in model.named_parameters():
            if not n_.startswith('fc.'):
                p_.requires_grad = False

    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=lr, momentum=0.9, nesterov=True, weight_decay=wd)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))

    last = os.path.join(CKPT_DIR, tag + '_last.pt')
    best_p = os.path.join(CKPT_DIR, tag + '_best.pt')
    start, best, hist = 0, 0.0, []
    if os.path.exists(last):
        ck = _load_ckpt(last)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
        start, best, hist = ck['epoch'] + 1, ck['best'], ck['hist']
        print('  resumed at epoch %d (best %.2f%%)' % (start, best))

    def set_lr(ep):
        f = (ep + 1) / warmup if ep < warmup else \
            0.5 * (1 + math.cos(math.pi * (ep - warmup) / max(1, epochs - warmup)))
        for g in opt.param_groups:
            g['lr'] = lr * f
        return lr * f

    print('[%s] split=%s  classes=%d  train=%d  test=%d'
          % (tag, split, num_classes, len(tr_idx), len(te_idx)))
    for ep in range(start, epochs):
        cur_lr = set_lr(ep); model.train(); t0 = time.time(); run = 0.0
        for xb, yb in tl:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
                loss = crit(model(xb), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            run += loss.item()
        acc = evaluate(model, vl)
        hist.append({'epoch': ep, 'loss': run / max(1, len(tl)), 'acc': acc, 'lr': cur_lr})
        torch.save({'epoch': ep, 'model': model.state_dict(), 'opt': opt.state_dict(),
                    'best': max(best, acc), 'hist': hist}, last)
        if acc > best:
            best = acc; torch.save({'model': model.state_dict(), 'acc': acc}, best_p)
        print('  ep %3d/%d  loss %.3f  acc %6.2f%%  best %6.2f%%  lr %.4f  (%.0fs)'
              % (ep + 1, epochs, hist[-1]['loss'], acc, best, cur_lr, time.time() - t0))

    json.dump(hist, open(os.path.join(RES_DIR, tag + '_history.json'), 'w'), indent=1)
    print('[%s] BEST %.2f%%' % (tag, best))
    return {'tag': tag, 'split': split, 'classes': num_classes,
            'best_acc': best, 'ckpt': best_p}

RESULTS = {}
RES_JSON = os.path.join(RES_DIR, 'phase1_results.json')
if os.path.exists(RES_JSON):
    RESULTS = json.load(open(RES_JSON))
    print('loaded previous results:', list(RESULTS))

def save_results():
    json.dump(RESULTS, open(RES_JSON, 'w'), indent=2)

## 1 · Checkpoint path resolver

The absolute paths stored in `phase1_results.json` were written before the
Drive account change, so several cells failed with `FileNotFoundError` on files
that do exist. This resolves every checkpoint by searching and repairs the
in-memory dictionary.

In [ ]:
import json, os, glob, inspect

RES_A = os.path.join(RES_DIR, 'revisionA')
os.makedirs(RES_A, exist_ok=True)
RA_PATH = os.path.join(RES_A, 'revisionA.json')
RA = json.load(open(RA_PATH)) if os.path.exists(RA_PATH) else {}
def save_ra():
    json.dump(RA, open(RA_PATH, 'w'), indent=2)

RESULTS = json.load(open(os.path.join(RES_DIR, 'phase1_results.json')))


def find_ckpt(tag):
    roots = [CKPT_DIR,
             '/content/drive/MyDrive/PhD_Q1/checkpoints',
             '/content/drive/MyDrive/PhD_Q1_own/checkpoints']
    for r in roots:
        if not os.path.isdir(r):
            continue
        for suf in ['_best.pt', '_last.pt']:
            p = os.path.join(r, tag + suf)
            if os.path.exists(p):
                return p
    hits = glob.glob('/content/drive/MyDrive/**/%s_best.pt' % tag, recursive=True)
    return hits[0] if hits else None


print('repairing stored paths')
for key in list(RESULTS):
    p = find_ckpt(RESULTS[key].get('tag', key))
    if p:
        RESULTS[key]['ckpt'] = p
    else:
        print('  NOT FOUND:', key)
print('done\n')

# pin the genuine ST-GCN class so backbone swaps can never recurse
if 'BASE_STGCN' not in globals() or not inspect.isclass(globals().get('BASE_STGCN')):
    assert inspect.isclass(STGCN), 'STGCN is not a class; re-run the ST-GCN cell'
    BASE_STGCN = STGCN
print('base backbone pinned:', BASE_STGCN)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

## 2 · M4 — three-seed ladder rungs

The R2 and R3 figures carrying the argument are single-seed. Three action
models already exist from the seed runs, so the probes can be repeated at no
training cost. The convex probe is deterministic given features, so its
variation comes from the backbone seed; the MLP probe additionally varies with
its own initialisation, which we also sweep.

In [ ]:
@torch.no_grad()
def plain_features(ckpt, idx, bs=256, factory=None):
    m = (factory or BASE_STGCN)(60).to(DEVICE)
    ck = _load_ckpt(ckpt)
    sd = ck.get('model', ck.get('state_dict', ck))
    sd = {k: v for k, v in sd.items() if not k.startswith('fc.')}
    missing, _ = m.load_state_dict(sd, strict=False)
    if len(missing) > 2:
        print('  warning: %d missing keys loading %s'
              % (len(missing), os.path.basename(ckpt)))
    m.eval()
    ld = torch.utils.data.DataLoader(SkeletonDS(X, SUBJ_LABEL, idx, False), bs,
                                     shuffle=False, num_workers=4)
    o = []
    for xb, _ in ld:
        with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
            f = m.extract(xb.to(DEVICE))
        if f.ndim > 2:
            f = f.mean(dim=tuple(range(2, f.ndim)))
        o.append(f.float().cpu().numpy())
    del m
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return np.concatenate(o, 0)


def ladder_probes(ckpt, split='reid_rand', factory=None, mlp_seeds=(1, 2, 3)):
    tr, te = SPLITS[split]
    tr, te = np.sort(tr), np.sort(te)
    Ftr = plain_features(ckpt, tr, factory=factory)
    Fte = plain_features(ckpt, te, factory=factory)
    ytr, yte = np.asarray(SUBJ_LABEL)[tr], np.asarray(SUBJ_LABEL)[te]
    sc = StandardScaler().fit(Ftr)
    Ztr, Zte = sc.transform(Ftr), sc.transform(Fte)

    r = {'linear': 100 * LogisticRegression(max_iter=3000).fit(Ztr, ytr).score(Zte, yte)}
    mlps = [100 * MLPClassifier(hidden_layer_sizes=(512, 256), max_iter=300,
                                early_stopping=True, n_iter_no_change=15,
                                random_state=s).fit(Ztr, ytr).score(Zte, yte)
            for s in mlp_seeds]
    r['mlp'] = float(np.mean(mlps))
    r['mlp_sd'] = float(np.std(mlps, ddof=1)) if len(mlps) > 1 else 0.0
    r['mlp_runs'] = mlps
    return r


SEED_CKPTS = {}
for name, tag in [('seed0', 'A_action_xsub'), ('seed1', 'A_action_xsub_s2'),
                  ('seed2', 'A_action_xsub_s3')]:
    p = find_ckpt(tag)
    if p:
        SEED_CKPTS[name] = p
        print('%-7s -> %s' % (name, os.path.basename(p)))
    else:
        print('%-7s -> missing (%s)' % (name, tag))
assert SEED_CKPTS, 'no action checkpoints found'

RA.setdefault('ladder_seeds', {})
for name, ck in SEED_CKPTS.items():
    if name in RA['ladder_seeds']:
        continue
    RA['ladder_seeds'][name] = ladder_probes(ck)
    r = RA['ladder_seeds'][name]
    print('%-7s R2 %.2f%%   R3 %.2f +/- %.2f%%' % (name, r['linear'], r['mlp'], r['mlp_sd']))
    save_ra()

print()
for fld, lab in [('linear', 'R2 linear probe'), ('mlp', 'R3 MLP probe')]:
    v = np.array([r[fld] for r in RA['ladder_seeds'].values()])
    if len(v) > 1:
        print('%-18s %.2f +/- %.2f  (n=%d backbone seeds)'
              % (lab, v.mean(), v.std(ddof=1), len(v)))
    else:
        print('%-18s %.2f  (n=1)' % (lab, v[0]))

## 2b · Record results whose checkpoints were lost

The adaptive-graph runs completed and were measured, but their checkpoints were
deleted by a cleanup cell that has since been removed. The numbers are recorded
here so the tables are complete. Retraining that backbone is optional and is
only needed to re-probe it or to release the weights.

In [ ]:
import glob

RA.setdefault('agcn_action', {'tag': 'A_action_agcn', 'best_acc': 87.29, 'ckpt': '',
                              'note': 'measured; checkpoint lost, retrain to reproduce'})
RA.setdefault('agcn_reid',   {'tag': 'B_reid_agcn',  'best_acc': 83.65, 'ckpt': '',
                              'note': 'measured; checkpoint lost, retrain to reproduce'})
RA.setdefault('agcn_ft',     {'tag': 'D_ft_agcn',    'best_acc': 74.20, 'ckpt': '',
                              'note': 'measured; checkpoint lost, retrain to reproduce'})
RA.setdefault('agcn_ladder', {'linear': 16.32, 'mlp': 30.44})
save_ra()
print('recorded. RA holds:', sorted(RA.keys()))

# remove any epoch-0 stub left by an interrupted restart
for f in glob.glob(os.path.join(CKPT_DIR, 'A_action_agcn_*.pt')):
    ck = torch.load(f, map_location='cpu')
    if ck.get('best', -1) < 10:
        os.remove(f); print('removed stub', os.path.basename(f))


def assert_finite(ckpt, name):
    sd = _load_ckpt(ckpt)
    sd = sd.get('model', sd)
    bad = [k for k, v in sd.items()
           if torch.is_tensor(v) and not torch.isfinite(v).all()]
    if bad:
        raise RuntimeError('%s diverged: %d non-finite tensors' % (name, len(bad)))
    print('  %s: weights finite' % name)


def smoke_train(factory, steps=30, lr=0.1, bs=32):
    m = factory(60).to(DEVICE)
    opt = torch.optim.SGD(m.parameters(), lr=lr, momentum=0.9, nesterov=True)
    crit = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))
    tr, _ = SPLITS['xsub']
    ld = torch.utils.data.DataLoader(SkeletonDS(X, ACT_LABEL, np.sort(tr), True),
                                     bs, shuffle=True, num_workers=2, drop_last=True)
    m.train(); losses = []
    for i, (xb, yb) in enumerate(ld):
        if i >= steps:
            break
        opt.zero_grad(set_to_none=True)
        with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
            loss = crit(m(xb.to(DEVICE)), yb.to(DEVICE))
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(m.parameters(), 5.0)
        scaler.step(opt); scaler.update()
        losses.append(loss.item())
    ok = all(np.isfinite(losses))
    print('  smoke test: %d steps, loss %.3f -> %.3f, finite: %s'
          % (len(losses), losses[0], losses[-1], ok))
    del m
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    if not ok:
        raise RuntimeError('smoke test produced a non-finite loss')
    return ok


def swap_gcn(model, builder):
    A = build_adjacency()
    n = 0
    for mod in model.modules():
        if isinstance(mod, STGCNBlock):
            old = mod.gcn
            mod.gcn = builder(old.conv.in_channels,
                              old.conv.out_channels // old.K, A)
            n += 1
    return n


def run_backbone(name, factory, action_tag, reid_tag, ft_tag):
    """Resumable. Skips any job already recorded; reports where it resumes."""
    global STGCN
    jobs = [(name + '_action', action_tag, ACT_LABEL, 'xsub', 60, 60, 0.1, None),
            (name + '_reid',  reid_tag,  SUBJ_LABEL, 'reid_rand', 40, 80, 0.1, None),
            (name + '_ft',    ft_tag,    SUBJ_LABEL, 'reid_rand', 40, 80, 0.05, 'action')]
    try:
        STGCN = factory
        for key, tag, lab, sp, nc, ep, lr, init in jobs:
            if key in RA:
                print('%-16s RECORDED (%.2f%%) - skipped' % (key, RA[key]['best_acc']))
                continue
            last = os.path.join(CKPT_DIR, tag + '_last.pt')
            if os.path.exists(last):
                ck = torch.load(last, map_location='cpu')
                print('%-16s RESUMING from epoch %d of %d (best %.2f%%)'
                      % (key, ck.get('epoch', -1) + 1, ep, ck.get('best', -1)))
            else:
                print('%-16s starting fresh' % key)
            kw = {'init_from': RA[name + '_action']['ckpt'],
                  'freeze': False} if init == 'action' else {}
            RA[key] = run_experiment(tag=tag, labels=lab, split=sp, num_classes=nc,
                                     epochs=ep, lr=lr, bs=64, **kw)
            assert_finite(RA[key]['ckpt'], key)
            save_ra()

        if name + '_ladder' in RA:
            print('%-16s RECORDED - skipped' % (name + '_ladder'))
        else:
            RA[name + '_ladder'] = ladder_probes(RA[name + '_action']['ckpt'],
                                                 factory=factory)
            save_ra()
    finally:
        STGCN = BASE_STGCN

    print('\n--- %s ---' % name)
    print('action      %.2f%%' % RA[name + '_action']['best_acc'])
    print('R5 scratch  %.2f%%' % RA[name + '_reid']['best_acc'])
    print('R2 linear   %.2f%%' % RA[name + '_ladder']['linear'])
    print('R3 MLP      %.2f%%' % RA[name + '_ladder']['mlp'])
    print('R4 finetune %.2f%%' % RA[name + '_ft']['best_acc'])

print('\nhelpers defined')

## 3 · M1 — channel-refined backbone (CTR-GCN topology)

The action model is already at epoch 60 and will be skipped. This resumes with
the 80-epoch re-identification run, then the fine-tuning attacker and probes.
About 2 hours.

In [ ]:
class ChannelRefinedGC(nn.Module):
    """Shared A_k refined per channel by a bounded function of joint differences."""
    def __init__(self, in_c, out_c, A, rel=None):
        super().__init__()
        self.K = A.shape[0]
        self.register_buffer('A', torch.from_numpy(A.copy()).float())
        rel = rel or max(out_c // 8, 8)
        self.pre = nn.ModuleList([nn.Conv2d(in_c, rel, 1) for _ in range(self.K)])
        self.expand = nn.ModuleList([nn.Conv2d(rel, out_c, 1) for _ in range(self.K)])
        self.conv = nn.ModuleList([nn.Conv2d(in_c, out_c, 1) for _ in range(self.K)])
        self.beta = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        y = 0
        for k in range(self.K):
            with torch.autocast('cuda', enabled=False):
                p = self.pre[k](x.float()).mean(dim=2)
                diff = torch.tanh(p.unsqueeze(-1) - p.unsqueeze(-2))
                R = torch.tanh(self.expand[k](diff))
            z = self.conv[k](x)
            y = y + torch.einsum('nctv,vw->nctw', z, self.A[k].to(x.dtype)) \
                  + self.beta.to(x.dtype) * torch.einsum('nctv,ncvw->nctw', z, R.to(x.dtype))
        return y.contiguous()


def make_ctrgcn(num_class):
    m = BASE_STGCN(num_class)
    print('  channel-refined: replaced %d graph convolutions'
          % swap_gcn(m, lambda i, o, A: ChannelRefinedGC(i, o, A)))
    return m


_t = make_ctrgcn(60)
with torch.no_grad():
    _o = _t(torch.zeros(2, IN_CH, T_LEN, NUM_NODE, MAX_PERSON))
print('forward OK:', tuple(_o.shape), '| finite:', bool(torch.isfinite(_o).all()),
      '| %.2f M params' % (sum(p.numel() for p in _t.parameters()) / 1e6))
del _t

run_backbone('ctrgcn', make_ctrgcn, 'A_action_ctrgcn', 'B_reid_ctrgcn', 'D_ft_ctrgcn')

## 5 · M1 — bone-stream modality

2s-AGCN and its successors obtain much of their gain from a second stream of
bone vectors, each joint replaced by its displacement from its parent. This
tests whether the ladder pattern depends on the input modality as well as on
the architecture. The backbone is unchanged ST-GCN.

In [ ]:
PARENT = {c: p for c, p in INWARD}
ROOT = [v for v in range(NUM_NODE) if v not in PARENT][0]

def to_bone(Xin, chunk=4000):
    assert Xin.shape[-1] == 3 and Xin.shape[-2] == NUM_NODE, Xin.shape
    out = np.zeros(Xin.shape, dtype=np.float16)
    for a in range(0, len(Xin), chunk):
        blk = np.asarray(Xin[a:a + chunk], dtype=np.float32)
        new = np.zeros_like(blk)
        for c, p in PARENT.items():
            new[..., c, :] = blk[..., c, :] - blk[..., p, :]
        new[..., ROOT, :] = blk[..., ROOT, :]      # root keeps its position
        out[a:a + chunk] = new.astype(np.float16)
    return out


BONE_PATH = '/content/ntu60_T64_bone.npy'
if os.path.exists(BONE_PATH):
    X_BONE = np.load(BONE_PATH, mmap_mode='r')
    print('loaded cached bone array', X_BONE.shape)
else:
    np.save(BONE_PATH, to_bone(X))
    X_BONE = np.load(BONE_PATH, mmap_mode='r')
    print('written', BONE_PATH, X_BONE.shape)

X_ORIG = X
try:
    X = X_BONE
    if 'bone_action' not in RA:
        RA['bone_action'] = run_experiment(tag='A_action_bone', labels=ACT_LABEL,
                                           split='xsub', num_classes=60,
                                           epochs=60, lr=0.1, bs=64)
        save_ra()
    if 'bone_reid' not in RA:
        RA['bone_reid'] = run_experiment(tag='B_reid_bone', labels=SUBJ_LABEL,
                                         split='reid_rand', num_classes=40,
                                         epochs=80, lr=0.1, bs=64)
        save_ra()
    if 'bone_ladder' not in RA:
        RA['bone_ladder'] = ladder_probes(RA['bone_action']['ckpt'])
        save_ra()
finally:
    X = X_ORIG
    print('input array restored')

print('\n--- bone stream ---')
print('action      %.2f%%' % RA['bone_action']['best_acc'])
print('R5 scratch  %.2f%%' % RA['bone_reid']['best_acc'])
print('R2 linear   %.2f%%' % RA['bone_ladder']['linear'])
print('R3 MLP      %.2f%%' % RA['bone_ladder']['mlp'])

## 6 · M3 — entropy-maximisation objective

Moon et al. deliberately maximise the *entropy* of the adversary's prediction
rather than its cross-entropy, because maximising cross-entropy can yield
confidently-wrong outputs from which the true label is recoverable by inversion.
Our sweep used cross-entropy. This runs their objective at the operating point
so the negative result cannot be attributed to the weaker formulation.

The trunk now minimises `CE(action) - alpha * H(adversary)` while the adversary
head still minimises its own cross-entropy.

In [ ]:
import math, time

def train_entropy_defended(alpha=1.0, epochs=60, lr=0.1, bs=64, wd=5e-4,
                           warmup=5, ramp=10, workers=4, adv_lr=1e-3,
                           tag='P2_entropy'):
    tr_idx, te_idx = SPLITS['xsub']
    tl = torch.utils.data.DataLoader(
        DualDS(X, ACT_LABEL, ID_LABEL_XSUB, tr_idx, True), bs, shuffle=True,
        num_workers=workers, pin_memory=(DEVICE == 'cuda'), drop_last=True)
    vl = torch.utils.data.DataLoader(
        DualDS(X, ACT_LABEL, ID_LABEL_XSUB, te_idx, False), bs * 2,
        shuffle=False, num_workers=workers)
    rng = np.random.RandomState(0)
    valid = np.asarray([i for i in tr_idx if ID_LABEL_XSUB[i] >= 0])
    avl = torch.utils.data.DataLoader(
        DualDS(X, ACT_LABEL, ID_LABEL_XSUB,
               np.sort(rng.choice(valid, min(4000, len(valid)), replace=False)), False),
        bs * 2, shuffle=False, num_workers=workers)

    model = DefendedSTGCN(60, N_ID_XSUB).to(DEVICE)
    trunk = [p for n_, p in model.named_parameters() if not n_.startswith('adv_head')]
    opt = torch.optim.SGD(trunk, lr=lr, momentum=0.9, nesterov=True, weight_decay=wd)
    opt_adv = torch.optim.Adam(model.adv_head.parameters(), lr=adv_lr)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    crit_adv = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))

    last = os.path.join(CKPT_DIR, tag + '_last.pt')
    best_p = os.path.join(CKPT_DIR, tag + '_best.pt')
    start, best, hist = 0, 0.0, []
    if os.path.exists(last):
        ck = _load_ckpt(last)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
        if 'opt_adv' in ck:
            opt_adv.load_state_dict(ck['opt_adv'])
        start, best, hist = ck['epoch'] + 1, ck['best'], ck['hist']
        print('  resumed at epoch %d' % start)
    adv = hist[-1]['adv'] if hist else float('nan')

    for ep in range(start, epochs):
        f = (ep + 1) / warmup if ep < warmup else \
            0.5 * (1 + math.cos(math.pi * (ep - warmup) / max(1, epochs - warmup)))
        for g in opt.param_groups:
            g['lr'] = lr * f
        a_now = alpha * min(1.0, (ep + 1) / max(1, ramp))
        model.train(); t0 = time.time(); H_run = 0.0

        for xb, ya, yi in tl:
            xb, ya, yi = xb.to(DEVICE), ya.to(DEVICE), yi.to(DEVICE)
            opt.zero_grad(set_to_none=True); opt_adv.zero_grad(set_to_none=True)
            with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
                feats = model.features(xb)
                logits_a = model.act_head(feats)
                # trunk step: maximise adversary entropy (no gradient reversal)
                logits_i = model.adv_head(feats)
                logp = torch.log_softmax(logits_i.float(), dim=1)
                H = -(logp.exp() * logp).sum(1).mean()
                loss = crit(logits_a, ya) - a_now * H
            scaler.scale(loss).backward()
            scaler.step(opt)
            # adversary step on detached features
            opt_adv.zero_grad(set_to_none=True)
            with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
                loss_adv = crit_adv(model.adv_head(feats.detach()), yi)
            scaler.scale(loss_adv).backward()
            scaler.step(opt_adv); scaler.update()
            H_run += H.item()

        model.eval(); corr = tot = ac = at = 0
        with torch.no_grad():
            for xb, ya, _ in vl:
                xb, ya = xb.to(DEVICE), ya.to(DEVICE)
                with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
                    oa, _ = model(xb, 0.0)
                corr += (oa.argmax(1) == ya).sum().item(); tot += ya.numel()
            for xb, _, yi in avl:
                xb, yi = xb.to(DEVICE), yi.to(DEVICE)
                with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
                    _, oi = model(xb, 0.0)
                ac += (oi.argmax(1) == yi).sum().item(); at += yi.numel()
        acc, adv = 100.0 * corr / tot, 100.0 * ac / max(1, at)
        hist.append({'epoch': ep, 'acc': acc, 'adv': adv, 'H': H_run / len(tl)})
        st = {'model': model.state_dict(), 'opt': opt.state_dict(),
              'opt_adv': opt_adv.state_dict(), 'epoch': ep,
              'best': max(best, acc), 'hist': hist}
        torch.save(st, last)
        if acc > best:
            best = acc; torch.save(st, best_p)
        print('  ep %2d/%d  act %.2f%%  adv %.2f%%  H %.3f  (%ds)'
              % (ep + 1, epochs, acc, adv, H_run / len(tl), time.time() - t0))

    print('[%s] BEST action %.2f%%  final adv %.2f%%' % (tag, best, adv))
    return {'tag': tag, 'alpha': alpha, 'action_acc': best,
            'adv_final': adv, 'ckpt': best_p}


if 'entropy_defence' not in RA:
    RA['entropy_defence'] = train_entropy_defended(alpha=1.0)
    save_ra()

# attack it with the full ladder
if 'entropy_ladder' not in RA:
    def defended_features(ckpt, idx, bs=256):
        m = DefendedSTGCN(60, N_ID_XSUB).to(DEVICE)
        m.load_state_dict(_load_ckpt(ckpt)['model']); m.eval()
        ld = torch.utils.data.DataLoader(
            DualDS(X, ACT_LABEL, ID_LABEL_XSUB, idx, False), bs,
            shuffle=False, num_workers=4)
        o = []
        with torch.no_grad():
            for xb, _, _ in ld:
                with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
                    o.append(m.features(xb.to(DEVICE)).float().cpu().numpy())
        del m
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
        return np.concatenate(o, 0)

    tr, te = SPLITS['reid_rand']
    tr, te = np.sort(tr), np.sort(te)
    Ftr, Fte = defended_features(RA['entropy_defence']['ckpt'], tr), \
               defended_features(RA['entropy_defence']['ckpt'], te)
    ytr, yte = np.asarray(SUBJ_LABEL)[tr], np.asarray(SUBJ_LABEL)[te]
    sc = StandardScaler().fit(Ftr)
    Ztr, Zte = sc.transform(Ftr), sc.transform(Fte)
    RA['entropy_ladder'] = {
        'linear': 100 * LogisticRegression(max_iter=3000).fit(Ztr, ytr).score(Zte, yte),
        'mlp': 100 * MLPClassifier(hidden_layer_sizes=(512, 256), max_iter=300,
                                   early_stopping=True, n_iter_no_change=15,
                                   random_state=1).fit(Ztr, ytr).score(Zte, yte)}
    save_ra()

if 'entropy_ft' not in RA:
    trunk_p = os.path.join(CKPT_DIR, 'P2_trunk_entropy.pt')
    sd = _load_ckpt(RA['entropy_defence']['ckpt'])['model']
    torch.save({'model': {k[len('trunk.'):]: v for k, v in sd.items()
                          if k.startswith('trunk.')}}, trunk_p)
    RA['entropy_ft'] = run_experiment(
        tag='P2_ft_entropy', labels=SUBJ_LABEL, split='reid_rand',
        num_classes=40, epochs=80, lr=0.05, bs=64, init_from=trunk_p, freeze=False)
    save_ra()

print('\n--- entropy objective (alpha = 1) ---')
print('action       %.2f%%   (cross-entropy lambda=1: 86.27%%)' % RA['entropy_defence']['action_acc'])
print('internal adv %.2f%%   (cross-entropy lambda=1: 16.88%%)' % RA['entropy_defence']['adv_final'])
print('R2 linear    %.2f%%   (cross-entropy lambda=1: 20.02%%)' % RA['entropy_ladder']['linear'])
print('R3 MLP       %.2f%%   (cross-entropy lambda=1: 36.43%%)' % RA['entropy_ladder']['mlp'])
print('R4 finetune  %.2f%%   (cross-entropy lambda=1: 78.95%%)' % RA['entropy_ft']['best_acc'])

## 7 · Assemble tables

In [ ]:
L = ['Table A1. Backbone and modality variants (random split, 40 identities, chance 2.50%)',
     '',
     '%-28s %9s %11s %9s %9s %10s' % ('variant', 'action', 'R5 scratch', 'R2', 'R3', 'R4'),
     '-' * 82,
     '%-28s %8.2f%% %10.2f%% %8.2f%% %8.2f%% %9.2f%%'
     % ('ST-GCN, joints (paper)', 86.85, 78.86, 18.67, 31.46, 79.89)]
for key, lab in [('agcn', 'adaptive graph, joints'),
                 ('ctrgcn', 'channel-refined, joints'),
                 ('bone', 'ST-GCN, bone stream')]:
    if key + '_action' in RA:
        L.append('%-28s %8.2f%% %10.2f%% %8.2f%% %8.2f%% %9s'
                 % (lab, RA[key + '_action']['best_acc'], RA[key + '_reid']['best_acc'],
                    RA[key + '_ladder']['linear'], RA[key + '_ladder']['mlp'],
                    ('%.2f%%' % RA[key + '_ft']['best_acc']) if key + '_ft' in RA else '-'))

if RA.get('ladder_seeds'):
    L += ['', 'Table A2. Ladder rungs over backbone seeds', '']
    for fld, lab in [('linear', 'R2 linear probe'), ('mlp', 'R3 MLP probe')]:
        v = np.array([r[fld] for r in RA['ladder_seeds'].values()])
        L.append('  %-20s %.2f +/- %.2f  (n=%d)' % (lab, v.mean(),
                 v.std(ddof=1) if len(v) > 1 else 0.0, len(v)))

if 'entropy_defence' in RA:
    L += ['', 'Table A3. Adversarial objective comparison at the operating point', '',
          '%-24s %9s %9s %9s %9s %9s' % ('objective', 'action', 'int adv', 'R2', 'R3', 'R4'),
          '-' * 74,
          '%-24s %8.2f%% %8.2f%% %8.2f%% %8.2f%% %8.2f%%'
          % ('cross-entropy (lam=1)', 86.27, 16.88, 20.02, 36.43, 78.95),
          '%-24s %8.2f%% %8.2f%% %8.2f%% %8.2f%% %8s'
          % ('entropy (alpha=1)', RA['entropy_defence']['action_acc'],
             RA['entropy_defence']['adv_final'], RA['entropy_ladder']['linear'],
             RA['entropy_ladder']['mlp'],
             ('%.2f%%' % RA['entropy_ft']['best_acc']) if 'entropy_ft' in RA else '-')]

txt = '\n'.join(L)
print(txt)
open(os.path.join(RES_A, 'revisionA_tables.txt'), 'w').write(txt)
save_ra()
print('\nwritten to', RES_A)

## 7 · Optional — retrain the adaptive-graph backbone

Only needed if you want the weights for the code release or a second seed. The
numbers in Table A1 are already recorded. About 2.5 hours; run it last.

In [ ]:
RETRAIN_AGCN = False          # set True only when everything else has finished

if RETRAIN_AGCN:
    class AdaptiveGC(nn.Module):
        def __init__(self, in_c, out_c, A, inter=None):
            super().__init__()
            self.K = A.shape[0]
            self.register_buffer('A', torch.from_numpy(A.copy()).float())
            self.B = nn.Parameter(torch.zeros(self.K, A.shape[1], A.shape[2]))
            self.eps = nn.Parameter(torch.tensor(0.1))
            inter = inter or max(out_c // 4, 16)
            self.theta = nn.ModuleList([nn.Conv2d(in_c, inter, 1) for _ in range(self.K)])
            self.phi = nn.ModuleList([nn.Conv2d(in_c, inter, 1) for _ in range(self.K)])
            self.conv = nn.ModuleList([nn.Conv2d(in_c, out_c, 1) for _ in range(self.K)])
            self.alpha = nn.Parameter(torch.zeros(1))
            self.inter = inter

        def forward(self, x):
            y = 0
            for k in range(self.K):
                with torch.autocast('cuda', enabled=False):
                    th = self.theta[k](x.float()).mean(dim=2)
                    ph = self.phi[k](x.float()).mean(dim=2)
                    logits = torch.bmm(th.transpose(1, 2), ph) / (self.inter ** 0.5)
                    Ck = torch.softmax(logits.clamp(-30, 30), dim=-1)
                Ak = (self.A[k] + self.eps * torch.tanh(self.B[k])).to(x.dtype)
                z = self.conv[k](x)
                y = y + torch.einsum('nctv,vw->nctw', z, Ak) \
                      + self.alpha.to(x.dtype) * torch.einsum('nctv,nvw->nctw', z, Ck.to(x.dtype))
            return y.contiguous()

    def make_agcn(num_class):
        m = BASE_STGCN(num_class)
        print('  adaptive: replaced %d graph convolutions'
              % swap_gcn(m, lambda i, o, A: AdaptiveGC(i, o, A)))
        return m

    smoke_train(make_agcn)
    for k in ['agcn_action', 'agcn_reid', 'agcn_ft', 'agcn_ladder']:
        RA.pop(k, None)
    save_ra()
    run_backbone('agcn', make_agcn, 'A_action_agcn', 'B_reid_agcn', 'D_ft_agcn')
else:
    print('AGCN retrain disabled; recorded numbers are used in Table A1')